In [1]:
# Load in merged data frame 
# copied and pasted from Data/Data Scripts/merge_station_and_event_data_script.py

import pandas as pd
import numpy as np
import os
import math

# What this does: Counts the frequency of non nan's for each feature for each station
# and organizes this information into Features_No_NAN_Counts.csv.

# Note: In the documentation (isd-format-document.pdf) sometimes a number 
# stands in for a nan value. We will deal with that if we choose such a 
# feature later on.


dirc = 'Data/OK City Station Data/Raw Data'
raw_station_data = os.listdir('Data/OK City Station Data/Raw Data') # list files in directory

station_csv_files = [file for file in raw_station_data if ('.csv' in file) and ('Station' in file)] # get csv files

station_list = [] # to hold station numbers as keys and number of csv files as entries

for file in station_csv_files:
    underscore_split = file.split('_')
    station_num = underscore_split[1].replace('.csv','')  
    station_list.append(int(station_num)) # station_num is a string
    


# Read in CSV's as pd.DataFrame's concat along axis = 0 

station_pd_dfs=[pd.read_csv(f"{dirc}/{file}").copy() for file in station_csv_files]
stations_df = pd.concat(station_pd_dfs, axis = 0)
new_df = stations_df.reset_index().drop(['index', 'Unnamed: 0'], axis =1).copy()



new_df['YEAR-MONTH-DAY'] = new_df['DATE'].apply(lambda r: r.split('T')[0])
new_df['TIME'] = new_df['DATE'].apply(lambda r: r.split('T')[1])
new_df.drop(['DATE'],axis=1,inplace=True)

# combined oklahoma tornadoes

path = 'Data/Storm Event Data/Cleaned Data/NEW_Oklahoma_Tornadoes_2000_2021.csv'
df2 = pd.read_csv(path).copy()
# Really will only care about 'BEGIN_DATE_TIME', 'END_DATE_TIME', 'BEGIN_LAT',
# 'END_LAT', 'BEGIN_LON', 'END_LON'

df2 = df2[
        [
        'BEGIN_DATE_TIME', 
        'END_DATE_TIME', 
        'BEGIN_LAT', 
        'END_LAT', 
        'BEGIN_LON', 
        'END_LON'
        ]
        ]

# columns BEGIN-DATE and BEGIN-TIME
new_df2 = df2.copy()
new_df2['BEGIN_DATE'] = df2['BEGIN_DATE_TIME'].apply(lambda r : r.split(' ')[0])
new_df2['BEGIN_TIME'] = df2['BEGIN_DATE_TIME'].apply(lambda r : r.split(' ')[1])
new_df2['END_DATE'] = df2['END_DATE_TIME'].apply(lambda r : r.split(' ')[0])
new_df2['END_TIME'] = df2['END_DATE_TIME'].apply(lambda r : r.split(' ')[1])

new_df2.drop(['BEGIN_DATE_TIME','END_DATE_TIME'],axis=1,inplace=True)

# Change format of 'BEGIN_DATE' and 'END_DATE' to match that of 'YEAR_MONTH_DAY' in station data

month_num = {
            'JAN':'01',
            'FEB':'02',
            'MAR':'03', 
            'APR':'04', 
            'MAY':'05', 
            'JUN':'06', 
            'JUL':'07', 
            'AUG':'08', 
            'SEP':'09', 
            'OCT':'10', 
            'NOV':'11', 
            'DEC':'12'
            }

new_df2['BEGIN_DATE'] = new_df2['BEGIN_DATE'].apply(lambda r: f'{r.split('-')[0]}-{month_num[r.split('-')[1]]}-{r.split('-')[2]}')
new_df2['END_DATE'] = new_df2['END_DATE'].apply(lambda r: f'{r.split('-')[0]}-{month_num[r.split('-')[1]]}-{r.split('-')[2]}')

# Get year to be 20**
new_df2['BEGIN_DATE']=new_df2['BEGIN_DATE'].apply(lambda r: f'20{r.split('-')[2]}-{r.split('-')[1]}-{r.split('-')[0]}')
new_df2['END_DATE']=new_df2['END_DATE'].apply(lambda r: f'20{r.split('-')[2]}-{r.split('-')[1]}-{r.split('-')[0]}')

data = pd.merge(left=new_df,right=new_df2,how ='outer',left_on='YEAR-MONTH-DAY',right_on='BEGIN_DATE')

/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_35868/1703931115.py:33: DtypeWarning: Columns (7,14,15,16,17,19,20,21,22,23,24,25,26,27,28,29,30,32,33,34,40,41,42,43,44,45,46,47,50,51,52,56,57,58,59,60,65,68,69,70,71,76,79,80,81,82,83,90,91,92,93,94,95,96,97,98,99,102,104,105,106,110,111,113,120,121,122,125) have mixed types. Specify dtype option on import or set low_memory=False.
  station_pd_dfs=[pd.read_csv(f"{dirc}/{file}").copy() for file in station_csv_files]
/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_35868/1703931115.py:33: DtypeWarning: Columns (39,40,41,42,43,47,48,52,53,54,55,57,58,59,60,61,65,70,71,76,77,88,89,106,108,109,110) have mixed types. Specify dtype option on import or set low_memory=False.
  station_pd_dfs=[pd.read_csv(f"{dirc}/{file}").copy() for file in station_csv_files]
/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_35868/1703931115.py:33: DtypeWarning: Columns (15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,32,34,40,41,4

In [2]:
def lat_lon_metric(lat_lon_1:tuple[float,float],lat_lon_2:tuple[float,float]):
    # Haversine Formula for computing the number of kilometers between 
    # To use this must convert lat lon from degrees to radians.
    Rd = 6371 # approx. Earth radius in km
    lat_1,lon_1 = lat_lon_1[0],lat_lon_1[1] # in deg
    lat_2,lon_2 = lat_lon_2[0],lat_lon_2[1] # in deg
    rlat_1 = (lat_1*(math.pi))/(180) # convert to radians
    rlon_1 = (lon_1*math.pi)/(180) # convert to radians
    rlat_2 = (lat_2*math.pi)/(180) # convert to radians
    rlon_2 = (lon_2*math.pi)/(180) # convert to radians
    Term_1 = (math.sin((rlat_2-rlat_1)/2))**2
    Term_2 = math.cos(rlat_1)*math.cos(rlat_2)*(math.sin((rlon_2-rlon_1)/2))**2
    a = Term_1+Term_2
    c = 2*math.atan2(math.sqrt(a),math.sqrt(1-a))
    D=Rd*c
    return D

def within_radius(lat_lon_1:tuple[float,float],lat_lon_2:tuple[float,float], valid_radius :float):
    if np.isnan(lat_lon_1[0]):
        return np.nan
    if np.isnan(lat_lon_1[0]):
        return np.nan
    if np.isnan(lat_lon_1[0]):
        return np.nan
    if np.isnan(lat_lon_1[0]):
        return np.nan
    
    great_circle_distance = lat_lon_metric(lat_lon_1,lat_lon_2)
    
    if great_circle_distance <= valid_radius:
        return True
    else:
        return False

In [3]:
# lines up with https://www.omnicalculator.com/other/latitude-longitude-distance
# example with Paris (48.8566,2.3522) and Krakow (50.0647,19.9450)
# Uses atan2 for more accuracy.
lat_lon_metric((48.8566,2.3522),(50.0647,19.9450)) 

1275.5699719152508

In [4]:
# Create Tornado indicator column
# Things to keep in mind:
# 1. At what time should a tornado be indicated? 
#       eg. during which time periods should we indicate? The same day? within n hours?
# 2. How far from the station should a tornado be in order to count for that station?


### HYPERPARAMETER time_window in hours
### if a tornado happens at 16:00 and time_window is 1,
### then a tornado is indicated at 15:00,16:00,17:00

time_window = 1 # hours

### HYPERPARAMETER valid_radius in km
### if a tornado occurs within valid_radius km of a station
### a tornado is indicated for that station within the time_window.

### NOTE: If we want improvements, we can draw a "spherical" lin
# between the starting and end lat,lon of a tornado.
### If this line crosses a valid radius of a station, 
### it will be counted. This ASSUMES TORNADO TAKES STRAIGHT LINE.

val_radius = 50 # km

# rename some columns 

rename = {
            'BEGIN_DATE':'TORNADO_BEGIN_DATE',
            'END_DATE':'TORNADO_END_DATE',
            'BEGIN_TIME': 'TORNADO_BEGIN_TIME',
            'END_TIME':'TORNADO_END_TIME',
            'BEGIN_LAT' : 'TORNADO_BEGIN_LAT',
            'END_LAT' : 'TORNADO_END_LAT',
            'BEGIN_LON' : 'TORNADO_BEGIN_LON',
            'END_LON' : 'TORNADO_END_LON',
            'LATITUDE' : 'STATION_LAT',
            'LONGITUDE' : 'STATION_LON',
            'TIME': 'STATION_TIME'
        }

data.rename(columns=rename,inplace=True)


In [5]:
# Tornado starting distance from station
def initial_tornado_to_station_distance(pd_row):
    station_lat_lon = pd_row['STATION_LAT'],pd_row['STATION_LON']
    initial_tornado_lat_lon = pd_row['TORNADO_BEGIN_LAT'],pd_row['TORNADO_BEGIN_LON']
    return lat_lon_metric(station_lat_lon,initial_tornado_lat_lon)
def apply_valid_radius(pd_row,valid_radius : float):
    station_lat_lon = pd_row['STATION_LAT'],pd_row['STATION_LON']
    initial_tornado_lat_lon = pd_row['TORNADO_BEGIN_LAT'],pd_row['TORNADO_BEGIN_LON']
    return within_radius(station_lat_lon, initial_tornado_lat_lon,valid_radius)

data['TORNADO_INITIAL_DISTANCE_FROM_STATION'] = data.apply(lambda r : initial_tornado_to_station_distance(r),axis =1)
data[f'TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_{val_radius}_km'] = data.apply(lambda r: apply_valid_radius(r,val_radius),axis =1)

In [6]:
# Apply time_window

def apply_time_window(pd_row, time_window,valid_radius):
    tornado_begin_time = pd_row['TORNADO_BEGIN_TIME']
    station_time = pd_row['STATION_TIME']
    within_radius_boolean = pd_row[f'TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_{valid_radius}_km']
    # type check below because np.nan is a float in this column
    if type(tornado_begin_time) is float and np.isnan(tornado_begin_time):
        return False
    if not within_radius_boolean:
        return False
    # All times are in HOUR:MIN:SECONDS 
    # Capture HOURs
    tornado_hour = int(tornado_begin_time.split(":")[0])
    station_hour = int(station_time.split(":")[0])
    if np.abs(station_hour - tornado_hour)<= time_window:
        return True
    else:
        return False
    

data['TORNADO_OCCURRENCE'] = data.apply(lambda r : apply_time_window(r, time_window,val_radius), axis = 1)

In [7]:
data

,STATION,NAME,STATION_LAT,STATION_LON,ELEVATION,SOURCE,REPORT_TYPE,CALL_SIGN,QUALITY_CONTROL,AA1,...,TORNADO_END_LAT,TORNADO_BEGIN_LON,TORNADO_END_LON,TORNADO_BEGIN_DATE,TORNADO_BEGIN_TIME,TORNADO_END_DATE,TORNADO_END_TIME,TORNADO_INITIAL_DISTANCE_FROM_STATION,TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_50_km,TORNADO_OCCURRENCE
0,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,SY-MT,OKC,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
2,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
3,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
4,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1626347,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1626348,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1626349,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1626350,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False


In [8]:
list(data.columns)

['STATION',
 'NAME',
 'STATION_LAT',
 'STATION_LON',
 'ELEVATION',
 'SOURCE',
 'REPORT_TYPE',
 'CALL_SIGN',
 'QUALITY_CONTROL',
 'AA1',
 'AA2',
 'AA3',
 'AA4',
 'AB1',
 'AD1',
 'AE1',
 'AG1',
 'AH1',
 'AH2',
 'AH3',
 'AH4',
 'AH5',
 'AH6',
 'AI1',
 'AI2',
 'AI3',
 'AI4',
 'AI5',
 'AI6',
 'AJ1',
 'AK1',
 'AL1',
 'AM1',
 'AN1',
 'AT1',
 'AT2',
 'AT3',
 'AT4',
 'AT5',
 'AT6',
 'AT7',
 'AT8',
 'AU1',
 'AU2',
 'AU3',
 'AU4',
 'AU5',
 'AW1',
 'AW2',
 'AW3',
 'AW4',
 'AW5',
 'AW6',
 'AW7',
 'AX1',
 'AX2',
 'AX3',
 'AX4',
 'AX5',
 'AX6',
 'CALL_SIGN.1',
 'CIG',
 'DEW',
 'ED1',
 'EQD',
 'GA1',
 'GA2',
 'GA3',
 'GA4',
 'GA5',
 'GA6',
 'GD1',
 'GD2',
 'GD3',
 'GD4',
 'GE1',
 'GF1',
 'GJ1',
 'GK1',
 'GP1',
 'GQ1',
 'GR1',
 'HL1',
 'IA1',
 'KA1',
 'KA2',
 'KA3',
 'KA4',
 'KB1',
 'KB2',
 'KB3',
 'KC1',
 'KC2',
 'KD1',
 'KD2',
 'KE1',
 'KG1',
 'KG2',
 'MA1',
 'MD1',
 'MF1',
 'MG1',
 'MH1',
 'MK1',
 'MV1',
 'MW1',
 'MW2',
 'MW3',
 'MW4',
 'MW5',
 'OC1',
 'OD1',
 'OE1',
 'OE2',
 'OE3',
 'QUALITY_CONTRO

In [9]:
### NOTE It looks like there are multiple occurrences of the same tornado in STORM EVENTS for single stations.


data[data['TORNADO_OCCURRENCE'] == True][['STATION','STATION_TIME','YEAR-MONTH-DAY','TORNADO_BEGIN_DATE','TORNADO_BEGIN_TIME','TORNADO_INITIAL_DISTANCE_FROM_STATION','TORNADO_OCCURRENCE']]

,STATION,STATION_TIME,YEAR-MONTH-DAY,TORNADO_BEGIN_DATE,TORNADO_BEGIN_TIME,TORNADO_INITIAL_DISTANCE_FROM_STATION,TORNADO_OCCURRENCE
43025,72353013967,15:00:00,2000-10-22,2000-10-22,16:36:00,46.576796,True
43031,72353013967,15:53:00,2000-10-22,2000-10-22,16:36:00,46.576796,True
43037,72353013967,16:00:00,2000-10-22,2000-10-22,16:36:00,46.576796,True
43043,72353013967,16:53:00,2000-10-22,2000-10-22,16:36:00,46.576796,True
43045,72353013967,17:00:00,2000-10-22,2000-10-22,18:14:00,10.369470,True
...,...,...,...,...,...,...,...
1617004,72354403954,21:53:00,2021-10-26,2021-10-26,22:54:00,31.662357,True
1617007,72354403954,22:53:00,2021-10-26,2021-10-26,22:54:00,31.662357,True
1617009,72354403954,22:53:00,2021-10-26,2021-10-26,23:36:00,37.973072,True
1617010,72354403954,23:53:00,2021-10-26,2021-10-26,22:54:00,31.662357,True


In [10]:
# DROP IRRELEVANT FEATURES

list(data.columns)


['STATION',
 'NAME',
 'STATION_LAT',
 'STATION_LON',
 'ELEVATION',
 'SOURCE',
 'REPORT_TYPE',
 'CALL_SIGN',
 'QUALITY_CONTROL',
 'AA1',
 'AA2',
 'AA3',
 'AA4',
 'AB1',
 'AD1',
 'AE1',
 'AG1',
 'AH1',
 'AH2',
 'AH3',
 'AH4',
 'AH5',
 'AH6',
 'AI1',
 'AI2',
 'AI3',
 'AI4',
 'AI5',
 'AI6',
 'AJ1',
 'AK1',
 'AL1',
 'AM1',
 'AN1',
 'AT1',
 'AT2',
 'AT3',
 'AT4',
 'AT5',
 'AT6',
 'AT7',
 'AT8',
 'AU1',
 'AU2',
 'AU3',
 'AU4',
 'AU5',
 'AW1',
 'AW2',
 'AW3',
 'AW4',
 'AW5',
 'AW6',
 'AW7',
 'AX1',
 'AX2',
 'AX3',
 'AX4',
 'AX5',
 'AX6',
 'CALL_SIGN.1',
 'CIG',
 'DEW',
 'ED1',
 'EQD',
 'GA1',
 'GA2',
 'GA3',
 'GA4',
 'GA5',
 'GA6',
 'GD1',
 'GD2',
 'GD3',
 'GD4',
 'GE1',
 'GF1',
 'GJ1',
 'GK1',
 'GP1',
 'GQ1',
 'GR1',
 'HL1',
 'IA1',
 'KA1',
 'KA2',
 'KA3',
 'KA4',
 'KB1',
 'KB2',
 'KB3',
 'KC1',
 'KC2',
 'KD1',
 'KD2',
 'KE1',
 'KG1',
 'KG2',
 'MA1',
 'MD1',
 'MF1',
 'MG1',
 'MH1',
 'MK1',
 'MV1',
 'MW1',
 'MW2',
 'MW3',
 'MW4',
 'MW5',
 'OC1',
 'OD1',
 'OE1',
 'OE2',
 'OE3',
 'QUALITY_CONTRO

In [11]:
columns_to_drop = ['NAME',
                    'SOURCE',
                    'REPORT_TYPE',
                    'CALL_SIGN',
                    'QUALITY_CONTROL',
                    'CALL_SIGN.1',
                    'QUALITY_CONTROL.1',
                    'REPORT_TYPE.1',
                    'SOURCE.1',
                    'AB1',
                    'AD1',
                    'AE1',
                    'AG1',
                    'AH1',
                    'AH2',
                    'AH3',
                    'AH4',
                    'AH5',
                    'AH6',
                    'AI1',
                    'AI2',
                    'AI3',
                    'AI4',
                    'AI5',
                    'AI6',
                    'AK1',
                    'AM1',
                    'AN1',
                    'AT1',
                    'AT2',
                    'AT3',
                    'AT4',
                    'AT5',
                    'AT6',
                    'AT7',
                    'AT8',
                    'AU1',
                    'AU2',
                    'AU3',
                    'AU4',
                    'AU5',
                    'AW1',
                    'AW2',
                    'AW3',
                    'AW4',
                    'AW5',
                    'AW6',
                    'AW7',
                    'AX1',
                    'AX2',
                    'AX3',
                    'AX4',
                    'AX5',
                    'AX6',
                    'ED1',
                    'EQD',
                    'GD1',
                    'GD2',
                    'GD3',
                    'GD4',
                    'GE1',
                    'GF1',
                    'IA1',
                    'KC1',
                    'KC2',
                    'KD1',
                    'KD2',
                    'KE1',
                    'MH1',
                    'MV1',
                    'MW1',
                    'MW2',
                    'MW3',
                    'MW4',
                    'MW5',
                    'OE1',
                    'OE2',
                    'OE3',
                    'REM',
                    'SA1',
                    'UA1',
                    'UG1',
                    'WA1'
                    ]
### Reasons to get rid of 
# 'NAME': already have an identifier column 'STATION'.
# 'SOURCE': this is just the source or sources used to create sample.
# 'REPORT_TYPE': denotes the type of geophysical surface observation.
# 'CALL_SIGN': call letters assigned to a weather station. We already have an identifier.
# 'QUALITY_CONTROl': For predicting tornadoes, this might not be super useful.
# though it may be good to keep in mind if so desired. One can erase all V01 
# entries (no quality control).
# 'CALL_SIGN.1': see CALL_SIGN.
# 'QUALITY_CONTROL.1' : see QUALITY_CONTROL.
# 'REPORT_TYPE.1': see REPORT_TYPE.
# 'SOURCE.1': see SOURCE.
# 'AB1': Liquid Precipitation Monthly total-- too long of a time scale.
# 'AD1: Liquid Precipitation Greatest Amount in 24 Hours, For the month -- too long of a time scale.
# 'AE1': Number of Days with Specific Amounts for Each Month -- Can be obtained through AA1-AA4
# 'AG1': 'Precipitation Estimated Observation -- not sure how this is different from AA1-AA4
# 'AH1'-- AH6' : Liquid Precipitation Maximum Short Duration, For The Month -- too long of a time scale.
# 'AI1 -- AI6' : Identical to 'AH1'--'AH6'
# 'AK1 Greatest Snow Depth on Ground for the Month
# 'AM1':
# 'AN1':
# 'AT1--AT8': Data leakage
# 'AU1--AU5': Data leakage 
# 'AW1--AW7': Data leakage
# 'AX1--AX6': Data leakage
# 'ED1': Runway Visibility
# 'EQD':
# 'GD1--GD4': Similar to GA1-GA6
# 'GE1': Similar to GA1-GA6 (may include later)
# 'GF1': Similar to GA1-GA6 (may include later)
# 'IA1': 
# 'KC1--KD2': too long of a time scale
# 'KE1': Extreme Temperatures, Number of Days Exceeding Criteria, For the Month -- too long of a time scale
# 'MH1': Atmospheric Pressure Observation for the month -- too long of a time scale.
# 'MK1' : See 'MH1'
# 'MV1 : Present Weather in Vicinity Observation -- Potential Data Leakage
# 'MW1--MW5' : Present Weather Observation -- Potential Data Leakage 
# 'OE1--OE3': Already have WND
# 'REM' : These are remarks
# 'SA1': Sea Surface Temperature 
# 'UA1'--'UG1' Marine Data?
# 'WA1'-- Platform Ice accretion ###

data.drop(columns = columns_to_drop, inplace = True)



In [12]:
# SPLIT TUPLES IN PARTICULAR COLUMNS

# These are ordered in the way their tuples are ordered

mapping = {
'AA1':['AA1- Liquid Precipitation- Period Quantity in Hours', 
       'AA1- Liquid Precipitation- Depth Dimension',
       'AA1- Liquid Precipitation- Condition Code',
       'AA1- Quality Code'],

'AA2': ['AA2- Liquid Precipitation- Period Quantity in Hours', 
       'AA2- Liquid Precipitation- Depth Dimension',
       'AA2- Liquid Precipitation- Condition Code',
       'AA2- Quality Code'],

'AA3' : ['AA3- Liquid Precipitation- Period Quantity in Hours', 
       'AA3- Liquid Precipitation- Depth Dimension',
       'AA3- Liquid Precipitation- Condition Code',
       'AA3- Quality Code'],

'AA4': ['AA4- Liquid Precipitation- Period Quantity in Hours', 
       'AA4- Liquid Precipitation- Depth Dimension',
       'AA4- Liquid Precipitation- Condition Code',
       'AA4- Quality Code'],

'AJ1' : ['AJ1- Snow Depth- Dimension',
       'AJ1- Snow Depth- Condition Code',
       'AJ1- Snow Depth- Quality Code',
       'AJ1- Snow Depth- Equivalent Water Depth Dimension',
       'AJ1- Snow Depth- Equivalent Water Condition Code',
       'AJ1- Snow Depth- Equivalent Water Condition Quality Code'],

'AL1' : ['AL1- Snow Accumulation- Period Quantity',
       'AL1- Snow Accumulation- Depth Dimension',
       'AL1- Snow Accumulation- Condition Code',
       'AL1- Snow Accumulation- Quality Code'],

'CIG': ['CIG- Sky Condition Observation- Ceiling Height Dimension',
       'CIG- Sky Condition Observation- Ceiling Quality Code',
       'CIG- Sky Condition Observation- Ceiling Determination Code',
       'CIG- Sky Condition Observation- Cavok Code'],

'DEW':['DEW- Air Temperature Observation- Dew Point Temperature',
       'DEW- Air Temperature Observation- Dew Point Quality Code'],

'GA1':['GA1- Sky Cover Layer- Coverage Code',
       'GA1- Sky Cover Layer- Coverage Quality Code',
       'GA1- Sky Cover Layer- Base Height Dimensions',
       'GA1- Sky Cover Layer- Base Height Quality Code',
       'GA1- Sky Cover Layer- Cloud Type Code',
       'GA1- Sky Cover Layer- Cloud Type Quality Code'],

'GA2':['GA2- Sky Cover Layer- Coverage Code',
       'GA2- Sky Cover Layer- Coverage Quality Code',
       'GA2- Sky Cover Layer- Base Height Dimensions',
       'GA2- Sky Cover Layer- Base Height Quality Code',
       'GA2- Sky Cover Layer- Cloud Type Code',
       'GA2- Sky Cover Layer- Cloud Type Quality Code'],

'GA3':['GA3- Sky Cover Layer- Coverage Code',
       'GA3- Sky Cover Layer- Coverage Quality Code',
       'GA3- Sky Cover Layer- Base Height Dimensions',
       'GA3- Sky Cover Layer- Base Height Quality Code',
       'GA3- Sky Cover Layer- Cloud Type Code',
       'GA3- Sky Cover Layer- Cloud Type Quality Code'],

'GA4':['GA4- Sky Cover Layer- Coverage Code',
       'GA4- Sky Cover Layer- Coverage Quality Code',
       'GA4- Sky Cover Layer- Base Height Dimensions',
       'GA4- Sky Cover Layer- Base Height Quality Code',
       'GA4- Sky Cover Layer- Cloud Type Code',
       'GA4- Sky Cover Layer- Cloud Type Quality Code'],

'GA5':['GA5- Sky Cover Layer- Coverage Code',
       'GA5- Sky Cover Layer- Coverage Quality Code',
       'GA5- Sky Cover Layer- Base Height Dimensions',
       'GA5- Sky Cover Layer- Base Height Quality Code',
       'GA5- Sky Cover Layer- Cloud Type Code',
       'GA5- Sky Cover Layer- Cloud Type Quality Code'],

'GA6':['GA6- Sky Cover Layer- Coverage Code',
       'GA6- Sky Cover Layer- Coverage Quality Code',
       'GA6- Sky Cover Layer- Base Height Dimensions',
       'GA6- Sky Cover Layer- Base Height Quality Code',
       'GA6- Sky Cover Layer- Cloud Type Code',
       'GA6- Sky Cover Layer- Cloud Type Quality Code'],


'GJ1':['GJ1- Sunshine Observation- Sunshine Duration Quantity',
       'GJ1- Sunshine Observation- Sunshine Duration Quality Code'],

'GK1': ['GK1- Sunshine Observation- Percent of Possible Sunshine Quantity',
       'GK1- Sunshine Observation- Percent of Possible Sunshine Quality Code'],

'GP1' : ['GP1- Modeled Solar Irradiance Section- Time Period in Minutes',
       'GP1- Modeled Solar Irradiance Section- Modeled Global Horizontal',
       'GP1- Modeled Solar Irradiance Section- Modeled Global Horizontal Source Flag',
       'GP1- Modeled Solar Irradiance Section- Modeled Global Horizontal Uncertainty',
       'GP1- Modeled Solar Irradiance Section- Modeled Direct Normal',
       'GP1- Modeled Solar Irradiance Section- Modeled Direct Normal Source Flag',
       'GP1- Modeled Solar Irradiance Section- Modeled Direct Normal Uncertainty',
       'GP1- Modeled Solar Irradiance Section- Modeled Diffuse Horizontal',
       'GP1- Modeled Solar Irradiance Section- Modeled Diffuse Horizontal Source Flag',
       'GP1- Modeled Solar Irradiance Section- Time Period in Minutes'],

'GQ1':['GQ1- Hourly Solar Angle Section- Hourly Solar Angle Time Period',
       'GQ1- Hourly Solar Angle Section- Hourly Mean Zenith Angle',
       'GQ1- Hourly Solar Angle Section- Hourly Mean Zenith Angle Quality Code',
       'GQ1- Hourly Solar Angle Section- Hourly Mean Azimuth Angle',
       'GQ1- Hourly Solar Angle Section- Hourly Mean Azimuth Angle Quality Code'],

'GR1':['GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation Time Period',
       'GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation on a Horizontal Surface',
       'GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation on a Horizontal Surface Quality Code',
       'GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation Normal to the Sun',
       'GR1- Extraterrestrial Radiation Section- Hourly Extraterrestrial Radiation Normal to the Sun Quality Code'],

'HL1':['HL1- Hail- Size',
       'HL1- Hail- Size Quality Code'],

'KA1': ['KA1- Extreme Air Temperature- Period Quantity',
       'KA1- Extreme Air Temperature- Code',
       'KA1- Extreme Air Temperature- Air Temperature',
       'KA1- Extreme Air Temperature- Temperature Quality Code'],
'KA2': ['KA2- Extreme Air Temperature- Period Quantity',
       'KA2- Extreme Air Temperature- Code',
       'KA2- Extreme Air Temperature- Air Temperature',
       'KA2- Extreme Air Temperature- Temperature Quality Code'],
'KA3': ['KA3- Extreme Air Temperature- Period Quantity',
       'KA3- Extreme Air Temperature- Code',
       'KA3- Extreme Air Temperature- Air Temperature',
       'KA3- Extreme Air Temperature- Temperature Quality Code'],
'KA4': ['KA4- Extreme Air Temperature- Period Quantity',
       'KA4- Extreme Air Temperature- Code',
       'KA4- Extreme Air Temperature- Air Temperature',
       'KA4- Extreme Air Temperature- Temperature Quality Code'],
'KB1': ['KB1- Average Air Temperature- Period Quantity',
       'KB1- Average Air Temperature- Type Code',
       'KB1- Average Air Temperature- Air Temperature',
       'KB1- Average Air Temperature- Temperature Quality Code'],
'KB2': ['KB2- Average Air Temperature- Period Quantity',
       'KB2- Average Air Temperature- Type Code',
       'KB2- Average Air Temperature- Air Temperature',
       'KB2- Average Air Temperature- Temperature Quality Code'],
'KB3': ['KB3- Average Air Temperature- Period Quantity',
       'KB3- Average Air Temperature- Type Code',
       'KB3- Average Air Temperature- Air Temperature',
       'KB3- Average Air Temperature- Temperature Quality Code'],

'KG1':['KG1- Average Dew Point and Wet Bulb Temperature- Period Quantity',
       'KG1- Average Dew Point and Wet Bulb Temperature- Code',
       'KG1- Average Dew Point and Wet Bulb Temperature- Temperature',
       'KG1- Average Dew Point and Wet Bulb Temperature- Derived Code',
       'KG1- Average Dew Point and Wet Bulb Temperature- Quality Code'],

'KG2':['KG2- Average Dew Point and Wet Bulb Temperature- Period Quantity',
       'KG2- Average Dew Point and Wet Bulb Temperature- Code',
       'KG2- Average Dew Point and Wet Bulb Temperature- Temperature',
       'KG2- Average Dew Point and Wet Bulb Temperature- Derived Code',
       'KG2- Average Dew Point and Wet Bulb Temperature- Quality Code'],

'MA1':['MA1-Atmospheric Pressure Observation- Altimeter Setting Rate',
       'MA1-Atmospheric Pressure Observation- Altimeter Quality Code',
       'MA1-Atmospheric Pressure Observation- Station Pressure Rate',
       'MA1-Atmospheric Pressure Observation- Station Pressure Quality Code'],

'MD1':['MD1- Atmospheric Pressure Change- Tendency Code',
       'MD1- Atmospheric Pressure Change- Quality Tendency Code',
       'MD1- Atmospheric Pressure Change- Three Hour Quantity',
       'MD1- Atmospheric Pressure Change- Quality Three Hour Code',
       'MD1- Atmospheric Pressure Change- Twenty Four Hour Quantity',
       'MD1- Atmospheric Pressure Change- Quality Twenty Four Hour Code'],

'MF1': ['MF1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure for the Day (Derived)',
       'MF1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure Quality Code',
       'MF1- Atmospheric Pressure Observation (SLP/STP)- Average Sea Level Pressure for the Day',
       'MF1- Atmospheric Pressure Observation (SLP/STP)- Average Sea Level Pressure Quality Code'],

'MG1': ['MG1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure for the Day (Derived)',
       'MG1- Atmospheric Pressure Observation (SLP/STP)- Average Station Pressure Quality Code',
       'MG1- Atmospheric Pressure Observation (SLP/STP)- Minimum Sea Level Pressure for the Day',
       'MG1- Atmospheric Pressure Observation (SLP/STP)- Minimum Sea Level Pressure Quality Code'],

'OC1':['OC1- Wind Gust Observation- Speed Rate',
       'OC1- Wind Gust Observation- Quality Code'],

'RH1':['RH1- Relative Humidity- Period Quantity',
       'RH1- Relative Humidity- Code',
       'RH1- Relative Humidity- Percentage',
       'RH1- Relative Humidity- Derived Code',
       'RH1- Relative Humidity- Quality Code'],

'RH2':['RH2- Relative Humidity- Period Quantity',
       'RH2- Relative Humidity- Code',
       'RH2- Relative Humidity- Percentage',
       'RH2- Relative Humidity- Derived Code',
       'RH2- Relative Humidity- Quality Code'],

'RH3':['RH3- Relative Humidity- Period Quantity',
       'RH3- Relative Humidity- Code',
       'RH3- Relative Humidity- Percentage',
       'RH3- Relative Humidity- Derived Code',
       'RH3- Relative Humidity- Quality Code'],

'SLP':['SLP- Atmospheric Pressure Observation- Sea Level Pressure',
       'SLP- Atmospheric Pressure Observation- Sea Level Pressure Quality Code'],

'TMP':['TMP- Air Temperature Observation- Air Temperature',
       'TMP- Air Temperature Observation- Air Temperature Quality Code'],

'VIS': ['VIS- Visibility Observation- Distance Dimension',
       'VIS- Visibility Observation- Distance Quality Code',
       'VIS- Visibility Observation- Variability Code',
       'VIS- Visibility Observation- Quality Variability Code'],

'WND':['WND- Wind Observation- Direction Angle',
       'WND- Wind Observation- Direction Quality Code',
       'WND- Wind Observation- Type Code',
       'WND- Wind Observation- Speed Rate',
       'WND- Wind Observation- Speed Quality Code'],
}




In [13]:
def split_tuple_features(df: pd.DataFrame,
                        mapping: dict,
                        drop_originals:bool =False,
                        tuple_sep:str =None) -> pd.DataFrame:
    """
    Splits specified tuple‐columns into separate descriptive columns.
    Args:
    df : pandas.DataFrame — your data.
    mapping : dict — key = column name, value = list of new sub‐column names.
    drop_original : bool — if True, drop the original tuple column after splitting.
    tuple_sep : str or None — if the tuple is stored as a string with a separator, provide the separator (e.g., ",").
    
    Returns:
    A new pandas DataFrame with the expanded columns.
    """
    def split_function(val,i:int):
        
    
        if pd.isna(val):
            return pd.isna(val)
        elif isinstance(val,tuple):
            return val[i]
        elif isinstance (val,str):
            return val.split(tuple_sep)[i]
        else:
            print(f'Problem with {val}')
            return val
        
    for original_col, new_cols in mapping.items():
        for i in range(0,len(new_cols)):
            df[new_cols[i]] = df[original_col].apply(lambda row : split_function(row,i))
    return df

In [14]:
split_tuple_features(data, mapping=mapping, tuple_sep=',')

/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_35868/1464598021.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_cols[i]] = df[original_col].apply(lambda row : split_function(row,i))
/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_35868/1464598021.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_cols[i]] = df[original_col].apply(lambda row : split_function(row,i))
/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_35868/1464598021.py:31: PerformanceWarning: DataFrame is highly

,STATION,STATION_LAT,STATION_LON,ELEVATION,AA1,AA2,AA3,AA4,AJ1,AL1,...,TMP- Air Temperature Observation- Air Temperature Quality Code,VIS- Visibility Observation- Distance Dimension,VIS- Visibility Observation- Distance Quality Code,VIS- Visibility Observation- Variability Code,VIS- Visibility Observation- Quality Variability Code,WND- Wind Observation- Direction Angle,WND- Wind Observation- Direction Quality Code,WND- Wind Observation- Type Code,WND- Wind Observation- Speed Rate,WND- Wind Observation- Speed Quality Code
0,72353013967,35.38890,-97.60060,391.7,"01,0000,9,5","06,0000,9,1",NaN,NaN,NaN,NaN,...,5,016000,5,N,1,130,5,N,0026,5
1,72353013967,35.38890,-97.60060,391.7,"01,0000,9,5",NaN,NaN,NaN,NaN,NaN,...,5,016000,5,N,1,150,5,N,0026,5
2,72353013967,35.38890,-97.60060,391.7,"01,0000,9,5",NaN,NaN,NaN,NaN,NaN,...,5,016000,5,N,1,140,5,N,0036,5
3,72353013967,35.38890,-97.60060,391.7,"01,0000,9,5",NaN,NaN,NaN,NaN,NaN,...,5,016000,5,N,1,180,5,N,0021,5
4,72353013967,35.38890,-97.60060,391.7,"01,0000,9,5",NaN,NaN,NaN,NaN,NaN,...,5,016000,5,N,1,150,5,N,0031,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1626347,72354403954,35.54113,-97.64725,390.2,"01,0000,9,5",NaN,NaN,NaN,NaN,NaN,...,5,016093,5,N,5,210,5,N,0077,5
1626348,72354403954,35.54113,-97.64725,390.2,"01,0000,9,5",NaN,NaN,NaN,NaN,NaN,...,5,016093,5,N,5,200,5,N,0093,5
1626349,72354403954,35.54113,-97.64725,390.2,"01,0000,9,5",NaN,NaN,NaN,NaN,NaN,...,5,016093,5,N,5,190,5,N,0057,5
1626350,72354403954,35.54113,-97.64725,390.2,"01,0000,9,5",NaN,NaN,NaN,NaN,NaN,...,5,016093,5,N,5,190,5,N,0046,5


In [15]:
# EXPLORE NA VALUES

In [16]:
# HANDLE NA VALUES

In [17]:
# SCATTER PLOTS